 ============================================================
# JUMIA NIGERIA - HOME & KITCHEN APPLIANCES SCRAPER
# Project: Jumia Market Intelligence Dashboard
# Phase 2: Web Scraping
# ============================================================

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random

print("✅ All libraries loaded!")

✅ All libraries loaded!


In [2]:
 # This "header" makes our scraper look like a real browser
# Without this, Jumia might block us ( definetly would block our request)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}

# The base URL for Jumia Small Appliances (blenders, kettles, microwaves, etc.)
# ?page=2 means page 2, ?page=3 means page 3, and so on
BASE_URL = "https://www.jumia.com.ng/small-appliances/?page={page}#catalog-listing"


In [3]:
# How many pages do you want to scrape?
# Start with 5 pages (~200 products) — you can increase later

TOTAL_PAGES = 5

print(f"⚙️  Settings ready. Will scrape {TOTAL_PAGES} pages.")


⚙️  Settings ready. Will scrape 5 pages.


In [4]:
# ---  Write a function to scrape ONE page ---

def scrape_page(page_number):
    """
    This function takes a page number,
    visits that page on Jumia,
    and returns a list of products found on that page.
    """


In [8]:
# --- STEP 3: Write a function to scrape ONE page ---
def scrape_page(page_number):
    """
    This function takes a page number,
    visits that page on Jumia,
    and returns a list of products found on that page.
    """
    # Build the URL for this page
    url = BASE_URL.format(page=page_number)
    print(f"\n🌐 Scraping page {page_number}: {url}")

    # Send a request to the URL (like opening it in a browser)
    try:
        response = requests.get(url, headers=HEADERS, timeout=10)
        if response.status_code != 200:
            print(f"  ⚠️  Page {page_number} returned status {response.status_code}. Skipping.")
            return []
    except requests.exceptions.RequestException as e:
        print(f"  ❌ Error loading page {page_number}: {e}")
        return []

    # Parse the HTML content of the page
    soup = BeautifulSoup(response.text, "lxml")

    # Find all product cards on the page
    product_cards = soup.find_all("article", class_="prd")
    print(f"  📦 Found {len(product_cards)} products on page {page_number}")

    # List to store all products from this page
    products = []

    # --- STEP 4: Loop through each product card and extract info ---
    for card in product_cards:

        # --- Product Name ---
        try:
            name = card.find("div", class_="name").text.strip()
        except AttributeError:
            name = "N/A"

        # --- Current Price ---
        try:
            price = card.find("div", class_="prc").text.strip()
        except AttributeError:
            price = "N/A"

        # --- Old Price (before discount) ---
        try:
            old_price = card.find("div", class_="old").text.strip()
        except AttributeError:
            old_price = "N/A"

        # --- Discount Percentage ---
        try:
            discount = card.find("div", class_="bdg _dsct").text.strip()
        except AttributeError:
            discount = "N/A"

        # --- Star Rating ---
        try:
            rating = card.find("div", class_="stars _s").text.strip()
        except AttributeError:
            rating = "N/A"

        # --- Number of Reviews ---
        try:
            reviews = card.find("div", class_="rev").text.strip()
        except AttributeError:
            reviews = "N/A"

        # --- Product Link ---
        try:
            link = "https://www.jumia.com.ng" + card.find("a", class_="core")["href"]
        except (AttributeError, TypeError):
            link = "N/A"

        # --- Save this product as a dictionary ---
        product = {
            "name":      name,
            "price":     price,
            "old_price": old_price,
            "discount":  discount,
            "rating":    rating,
            "reviews":   reviews,
            "link":      link,
            "page":      page_number
        }
        products.append(product)

    return products

In [10]:
# --- STEP 5: Scrape ALL pages ---

all_products = []   # Master list to hold everything

for page in range(1, TOTAL_PAGES + 1):

    # Scrape this page
    page_products = scrape_page(page)

    # Add the results to our master list
    all_products.extend(page_products)

    # Wait between 2 to 4 seconds before loading the next page
    # This is important — it prevents Jumia from blocking us
    
    wait_time = random.uniform(2, 4)
    print(f"  ⏳ Waiting {wait_time:.1f} seconds before next page...")
    time.sleep(wait_time)



🌐 Scraping page 1: https://www.jumia.com.ng/small-appliances/?page=1#catalog-listing
  📦 Found 55 products on page 1
  ⏳ Waiting 3.3 seconds before next page...

🌐 Scraping page 2: https://www.jumia.com.ng/small-appliances/?page=2#catalog-listing
  📦 Found 40 products on page 2
  ⏳ Waiting 2.8 seconds before next page...

🌐 Scraping page 3: https://www.jumia.com.ng/small-appliances/?page=3#catalog-listing
  📦 Found 40 products on page 3
  ⏳ Waiting 2.9 seconds before next page...

🌐 Scraping page 4: https://www.jumia.com.ng/small-appliances/?page=4#catalog-listing
  📦 Found 40 products on page 4
  ⏳ Waiting 3.5 seconds before next page...

🌐 Scraping page 5: https://www.jumia.com.ng/small-appliances/?page=5#catalog-listing
  📦 Found 40 products on page 5
  ⏳ Waiting 2.3 seconds before next page...


In [11]:
# --- STEP 6: Save the raw data to a CSV file ---

print(f"\n✅ Scraping complete! Total products collected: {len(all_products)}")

# Convert our list of dictionaries into a pandas DataFrame (like an Excel table)
df = pd.DataFrame(all_products)

# Preview the first 5 rows
print("\n📋 Preview of scraped data:")
print(df.head())

# Show the shape (rows, columns)
print(f"\n📊 Data shape: {df.shape[0]} rows × {df.shape[1]} columns")

# Save to CSV in the data/raw folder
output_path = "../data/raw/jumia_kitchen_raw.csv"
df.to_csv(output_path, index=False)

print(f"\n💾 Raw data saved to: {output_path}")
print("🎉 Phase 2 Complete! Now move on to 02_cleaning.ipynb")



✅ Scraping complete! Total products collected: 215

📋 Preview of scraped data:
                                                name      price old_price  \
0            Hisense WSPA503 WM 5kg Twin Tub Machine  ₦ 170,000       N/A   
1              3 Litres Whistling Gas & Stove Kettle   ₦ 12,000       N/A   
2  Hisense 5kg Manual Washing Machine Twin Tub Wi...  ₦ 162,315       N/A   
3  Binatone 3 Litres Electric Jug (CEJ-3000) - Bl...   ₦ 31,919       N/A   
4  Legacy 8L Yampounder And Multifunctional Food ...   ₦ 46,500       N/A   

  discount rating reviews                                               link  \
0      32%    N/A     N/A  https://www.jumia.com.ng/hisense-wspa503-wm-5k...   
1      N/A    N/A     N/A  https://www.jumia.com.ng/3-litres-whistling-ga...   
2      26%    N/A     N/A  https://www.jumia.com.ng/hisense-5kg-manual-wa...   
3      36%    N/A     N/A  https://www.jumia.com.ng/binatone-3-litres-ele...   
4      15%    N/A     N/A  https://www.jumia.com.ng/8l-ya